In [1]:
from keras.datasets import california_housing

# Make sure to pass version="small" to get the right dataset.
(train_data, train_targets), (test_data, test_targets) = (
    california_housing.load_data(version="small")
)

I0000 00:00:1785093999.986943    9784 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1785094000.575711    9784 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785094002.178874    9784 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


743530/743530 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step


# Regression: Predicting California Median House Values

**Task:** given 8 numeric attributes of a California district (location, age of housing, room counts, population, income, etc.), predict its **median house value** — a continuous dollar amount, not a category. This is a fundamentally different kind of problem from the [binary](./notebooks/binary_classification.ipynb) and [multiclass](./notebooks/multiclassification.ipynb) classification notebooks: there's no fixed set of buckets to sort examples into, so the network's output layer, loss function, and evaluation metric all need to change shape.

**Dataset:** the California Housing dataset (the "small" variant, 600 total districts: 480 train / 120 test) — deliberately tiny compared to IMDB's 50,000 reviews or Reuters' ~11,000 newswires. That smallness is the reason this notebook introduces a new technique, **K-fold cross-validation**, further down: with so little data, a single train/validation split would give a noisy, unreliable read on how well the model generalizes.

**What's structurally new here, previewed:**
- Output layer: a single unit with **no activation function** (linear), since a house value can be any real number — not squashed into `(0, 1)` like a probability.
- Loss: **mean squared error (MSE)**, the standard loss for regression, instead of any flavor of cross-entropy.
- Metric: **mean absolute error (MAE)**, reported directly in the target's units (here, "hundreds of thousands of dollars" — see the target-scaling cell below) because it's more human-interpretable than MSE.
- Validation strategy: **K-fold cross-validation** instead of one held-out slice.

In [ ]:
print(train_data.shape)
print(test_data.shape)

480 training examples and 8 features per example — small enough that this whole notebook trains in seconds, but also small enough that how you validate the model (see the K-fold section below) matters a lot more than it did for the classification notebooks' tens of thousands of examples.

In [ ]:
train_targets

`train_targets` is the **regression target**: the actual median house value for each district, in raw dollars (e.g., `228400.0`). Contrast this with classification labels, which were discrete class indices (`0`/`1`, or `0`–`45`) with no inherent numeric magnitude — here the numbers *are* the thing we're trying to predict, and "off by $5,000" is meaningfully better than "off by $200,000." That's precisely what a regression loss (MSE, below) is built to measure and what a classification loss (cross-entropy) is not.

In [7]:
train_data[0]

array([-1.2224e+02,  3.7730e+01,  2.1000e+01,  7.0310e+03,  1.2490e+03,
        2.9300e+03,  1.2350e+03,  4.5213e+00], dtype=float32)

Look at the wildly different scales of these 8 raw features: longitude `≈ -122`, latitude `≈ 38`, housing median age `≈ 21`, total rooms `≈ 7031`, total bedrooms `≈ 1249`, population `≈ 2930`, households `≈ 1235`, median income `≈ 4.5`. Feeding these directly into a network is a problem: a feature ranging in the thousands (`total_rooms`) would dominate the dot products in the first `Dense` layer purely because of its scale, not because it's actually more predictive than median income (which ranges 0-15ish). Gradient descent also struggles when features live on very different scales — it ends up having to use a tiny learning rate for the large-scale features and takes forever to make progress on the small-scale ones. The next cell fixes this.

In [8]:
mean = train_data.mean(axis=0)
std = train_data.std(axis=0)
x_train = (train_data - mean) / std
x_test = (test_data - mean) / std


## Feature normalization (standardization)

Each feature column is transformed to have **mean 0 and standard deviation 1**: `x' = (x - mean) / std`. After this, every feature lives on a comparable scale, which is standard practice before feeding numeric features into a neural network — it makes the loss surface much better-conditioned for gradient descent to navigate, and prevents large-magnitude features from dominating early training purely due to units (dollars vs. degrees of latitude vs. room counts) rather than actual predictive value.

**Important detail:** `mean` and `std` are computed **only from `train_data`**, and that *same* mean/std is then used to normalize `test_data` too — the test set's own statistics are never touched. This avoids **data leakage**: if you computed the test set's mean/std separately (or computed statistics over train+test combined), information about the test distribution would leak into preprocessing, making test performance look artificially better than what you'd get on truly unseen data in production.

In [9]:
y_train = train_targets / 100000
y_test = test_targets / 100000

The targets get a simpler rescaling: dividing by 100,000 turns raw dollar values like `228400.0` into `2.284`. This isn't standardization (no mean-centering, no dividing by std) — it's just a convenient unit change to "hundreds of thousands of dollars," keeping target values in a small range near 1-5 rather than the hundred-thousands. Networks generally train a bit more smoothly when the values they're predicting aren't extremely large, since huge targets produce huge initial losses/gradients relative to typically-small initial weights. It also means the MAE reported later can be read at a glance: an MAE of `0.5` means "off by about $50,000 on average."

In [ ]:
import keras
from keras import layers

def get_model():
    # Because you need to instantiate the same model multiple times,
    # you use a function to construct it.
    model = keras.Sequential(
        [
            layers.Dense(64, activation="relu"),
            layers.Dense(64, activation="relu"),
            layers.Dense(1),
        ]
    )
    model.compile(
        optimizer="adam",
        loss="mean_squared_error",
        metrics=["mean_absolute_error"],
    )
    return model

## Building the model: a linear (activation-free) output layer

```python
layers.Dense(1)
```

Note there's no `activation=` argument on the final layer — this is deliberate, and it's the key structural difference from every model in the classification notebooks. `sigmoid` forces its output into `(0, 1)`; `softmax` forces its outputs into a probability distribution. Both are appropriate when the model is predicting a probability. Here it's predicting a house value, which can be any real number in principle (and after our /100000 rescaling, is typically somewhere around 0.5-5, but nothing forces it there) — squashing it through `sigmoid`/`softmax` would artificially cap what the model could ever output, no matter how the training data actually looks. So the final layer is left as a **plain linear transformation**: `output = dot(W, input) + b`, free to produce any real number.

**Loss — `mean_squared_error` (MSE):** `loss = mean((y_true - y_pred)^2)`. Squaring the error does two things: it makes all errors non-negative (so overestimates and underestimates don't cancel out), and it penalizes large errors disproportionately more than small ones — a prediction off by $200,000 contributes 4x the loss of one off by $100,000, not 2x. That heavily-punish-large-errors behavior is what MSE is for, and it's the standard default loss for regression, playing the same "this is the loss gradient descent should minimize" role that cross-entropy played for classification.

**Metric — `mean_absolute_error` (MAE):** `metric = mean(|y_true - y_pred|)`. MAE is tracked purely for human interpretation (same "loss vs. metric" split we saw in the classification notebooks) — because it doesn't square the error, it stays in the original units of the target and is much easier to reason about than MSE's squared-dollars. An MAE of `0.5` (in our /100000-scaled units) directly means "this model's predictions are off by about $50,000 on average."

**Why wrap this in a `get_model()` function instead of building `model` once at the top level?** Because of the K-fold cross-validation below: each fold needs its own model trained **from scratch** (freshly initialized weights), so that one fold's training doesn't carry over into the next fold's — exactly the weight-carryover pitfall flagged in the binary classification notebook, but engineered around correctly this time by re-instantiating the model on every loop iteration.

In [ ]:
import numpy as np

k = 4
num_val_samples = len(x_train) // k
num_epochs = 4
all_mae_histories = []
for i in range(k):
    print(f"Processing fold #{i + 1}")
    # Prepares the validation data: data from partition #k
    fold_x_val = x_train[i * num_val_samples : (i + 1) * num_val_samples]
    fold_y_val = y_train[i * num_val_samples : (i + 1) * num_val_samples]
    # Prepares the training data: data from all other partitions
    fold_x_train = np.concatenate(
        [x_train[: i * num_val_samples], x_train[(i + 1) * num_val_samples :]],
        axis=0,
    )
    fold_y_train = np.concatenate(
        [y_train[: i * num_val_samples], y_train[(i + 1) * num_val_samples :]],
        axis=0,
    )
    # Builds the Keras model (already compiled)
    model = get_model()
    # Trains the model
    history = model.fit(
        fold_x_train,
        fold_y_train,
        validation_data=(fold_x_val, fold_y_val),
        epochs=num_epochs,
        batch_size=16,
        verbose=0,
    )
    mae_history = history.history["val_mean_absolute_error"]
    all_mae_histories.append(mae_history)

## K-fold cross-validation: why, and how it works

With only 480 training examples, a single 80/20-style train/validation split would set aside just ~100 examples for validation. The resulting validation MAE would depend heavily on *which* 100 examples happened to land in that slice — a different random split could easily shift the validation score by a noticeable margin, making it hard to trust any single number as "the" validation performance, or to reliably compare one model configuration against another.

**K-fold cross-validation** (`k=4` here) fixes this by not relying on just one split:
1. Partition the training data into `k` equal-sized chunks ("folds").
2. For each fold `i`: train a **fresh** model (`get_model()`, weights reinitialized) on the other `k - 1` folds, and validate on fold `i`.
3. This produces `k` independent validation scores — one per fold, since each fold gets a turn being the held-out set. Every example is used for both training and validation across the `k` runs, just never in the same run.

Averaging those `k` scores (done in the next cell) gives a much more stable estimate of generalization performance than any single split would, at the cost of training the model `k` times instead of once — a trade-off that's only worth it because the dataset is small enough that `k=4` full training runs are still cheap.

Note `num_epochs=4` is quite small — enough to see whether validation MAE is still trending down, but not enough to reach the "validation loss turns around and rises" overfitting point seen in the classification notebooks. Chollet's original version of this exercise uses ~500 epochs (with a smaller, faster architecture) specifically to make that overfitting turn visible; this notebook's 4 epochs mainly demonstrate the K-fold *mechanism* rather than a full overfitting curve.

In [16]:
average_mae_history = [
    np.mean([x[i] for x in all_mae_histories]) for i in range(num_epochs)
]

NameError: name 'all_mae_histories' is not defined

`all_mae_histories` is a list of `k` lists, each holding one fold's validation MAE per epoch (shape conceptually `k × num_epochs`). This line transposes that: for each epoch index `i`, it averages the `i`-th-epoch validation MAE **across all `k` folds**, producing a single `average_mae_history` curve of length `num_epochs` — "what was the typical validation MAE after training for `i` epochs, across all 4 independent train/validation splits."

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(average_mae_history) + 1)
plt.plot(epochs, average_mae_history)
plt.xlabel("Epochs")
plt.ylabel("Validation MAE")
plt.show()

This plots the cross-validated MAE against training length. In a regression setting, this is the analogue of the training/validation loss-divergence plots from the classification notebooks — except here, since only the *validation* curve (averaged across folds) is plotted, what you're looking for is whether it's still decreasing (train longer) or has flattened/started rising (stop around there, or add regularization). With just 4 epochs, expect this curve to mostly still be trending down, not yet showing a clear overfitting turn.

**Natural next steps, not shown in this notebook:** once you've used the K-fold curve to pick a good number of epochs, you'd typically retrain **one final model** on *all* of `x_train`/`y_train` (no folds held out) for that chosen epoch count, then evaluate it once on `x_test`/`y_test` — the test set hasn't been touched by any of the cross-validation above, so it remains a clean, unbiased final check.